## OpenAI — RAG investigation

This notebook is to understand this feature to apply this understanding to address an issue in Dimagi Open Chat Studio for the Remote Index feature. https://deepwiki.com/dimagi/open-chat-studio/7-document-collections-and-rag#remoteindexmanager-openai

File search is a tool available in the Responses API. 
It enables models to retrieve information in a knowledge base of previously uploaded files through semantic and keyword search. 
This is a hosted tool managed by OpenAI
When the model decides to use it, it will automatically call the tool, retrieve information from your files, and return an output.
https://developers.openai.com/api/docs/guides/tools-file-search

Must run all these steps together

### Step 1 - Upload a file to the File API

In [ ]:
from openai import OpenAI

client = OpenAI()

# Function to upload into OpenAI's file storage from a local file


def create_file(client, local_file_path):
    with open(local_file_path, "rb") as file_content:
        result = client.files.create(file=file_content, purpose="assistants")
    print(result.id)
    return result.id


# use example files in repo
file_id = create_file(client, "../docs/Example KBase doc.pdf")

### Step 2 - Create OpenAI vector store and upload the file to it

In [ ]:
kbase_vector_store = client.vector_stores.create(name="knowledge_base")
print(kbase_vector_store.id)

result = client.vector_stores.files.create(
    vector_store_id=kbase_vector_store.id, file_id=file_id
)
print(result)

### Step 3 - include the file_search tool & vector stores in which to search.

In [ ]:
my_vector_store_id = result.vector_store_id

response = client.responses.create(
    model="gpt-4.1-mini",
    input="Concisely summarize the document in 3 bullet points.",
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [my_vector_store_id],
            "max_num_results": 2,
        }
    ],
)
# print(response)
print(response.output_text)